# 📊 01 - Phân Tích Khám Phá Dữ Liệu (Exploratory Data Analysis - EDA)
### Đề tài: AI-Generated Text Detection Using Machine Learning & NLP with Chrome Extension

---
## 🎯 Mục tiêu của Notebook:
1. **Kiểm toán chất lượng dữ liệu (Data Quality Audit):** Đánh giá số lượng, tỷ lệ nhãn, kiểm tra văn bản thiếu, quá ngắn và bài viết trùng lặp trên file gốc `AI_Human.csv`.
2. **Lấy mẫu phân tầng an toàn (Stratified Chunk Sampling):** Đọc dữ liệu theo khối để lấy 60,000 mẫu đại diện phục vụ phân tích mà không làm tràn bộ nhớ RAM.
3. **Trích xuất đặc trưng phong cách (Stylometric Feature Engineering):** Đo lường cấu trúc ký tự, độ dài câu/từ và độ phong phú vốn từ vựng (Lexical Diversity).
4. **Phân tích biến gây nhiễu (Confounding Factor Analysis):** Kiểm tra tác động của độ dài văn bản lên nhãn phân loại.
5. **Xếp hạng mức độ phân biệt (Cohen's $d$ Effect Size):** Đo lường định lượng xem đặc trưng nào phân biệt AI và Human mạnh nhất.
6. **Thí nghiệm kiểm chứng có đối chứng (Matched Length Subsetting):** Khống chế độ dài bài viết bằng nhau để tìm ra các đặc trưng phân biệt bản chất.
7. **Khám phá dấu vân tay từ vựng (Discriminative N-grams):** Tìm top các từ và cụm từ đặc trưng nhất của AI và Người qua chênh lệch điểm TF-IDF.
8. **Đúc kết nguyên tắc tiền xử lý & Định hướng cho 4 Pipeline mô hình.**


---
## 1. Thiết lập Môi trường & Cấu hình Đồ họa
Cấu hình đồ họa đồng nhất với bảng màu chuyên dụng:
- 🔵 **Human:** `#2878B5` (Xanh dương)
- 🔴 **AI:** `#D9534F` (Đỏ cam)


In [ ]:
from collections import Counter
from pathlib import Path
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')

# Cấu hình phong cách biểu đồ trực quan
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 115,
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

COLOR_HUMAN = '#2878B5'
COLOR_AI = '#D9534F'
PALETTE = {'Human': COLOR_HUMAN, 'AI': COLOR_AI}
RANDOM_STATE = 42

# Tự động nhận diện đường dẫn file AI_Human.csv
possible_paths = [
    Path("../data/AI_Human.csv"),
    Path("data/AI_Human.csv"),
    Path("AI_Human.csv"),
    Path(r"D:\Project SamSung\AI-gen-text-detection\data\AI_Human.csv")
]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("Không tìm thấy file AI_Human.csv. Vui lòng kiểm tra lại thư mục data!")

print(f"✅ Đã nạp thành công các thư viện nền tảng!")
print(f"📁 Đường dẫn dữ liệu: {DATA_PATH.resolve()}")


---
## 2. Khảo sát Cấu trúc Ban đầu của Tập Dữ liệu
Đọc thử 5 dòng đầu tiên để kiểm tra định dạng các cột và nhãn nhị phân.


In [ ]:
df_preview = pd.read_csv(DATA_PATH, nrows=5)
print("Tên các cột trong dữ liệu:")
print(df_preview.columns.tolist())
print("\n5 dòng dữ liệu mẫu:")
display(df_preview)


---
## 3. Kiểm toán Chất lượng Dữ liệu & Lấy mẫu Phân tầng (Stratified Sampling)
Do file gốc `AI_Human.csv` chứa gần **500,000 bài viết** (~1.1 GB), việc nạp toàn bộ vào RAM để tính chuỗi ký tự sẽ dễ gây tràn bộ nhớ.
Ta áp dụng thuật toán **2 lượt đọc theo khối (Two-Pass Chunking)**:
- **Pass 1:** Quét qua từng khối (`chunksize=50_000`) để kiểm toán lỗi (rỗng, quá ngắn, nhãn sai) và đếm tổng số mẫu của từng lớp.
- **Pass 2:** Lấy mẫu ngẫu nhiên có trọng số từ từng khối để trích xuất tập con $N = 60,000$ dòng bảo toàn **chính xác tỷ lệ nhãn gốc**.


In [ ]:
def valid_rows(chunk):
    """Lọc các dòng hợp lệ: text không rỗng, độ dài > 10 ký tự, nhãn là 0 hoặc 1."""
    text = chunk['text'].astype('string')
    label = pd.to_numeric(chunk['generated'], errors='coerce')
    mask = text.notna() & text.str.strip().str.len().gt(10) & label.isin([0, 1])
    return pd.DataFrame({
        'text': text[mask].str.strip(),
        'generated': label[mask].astype('int8')
    })


def inspect_and_sample(path, sample_size=60_000, chunksize=50_000, seed=42):
    counts = Counter()
    audit = Counter()

    # Pass 1: Kiểm toán dữ liệu và đếm số lượng từng lớp
    for chunk in pd.read_csv(path, usecols=['text', 'generated'], chunksize=chunksize):
        audit['raw_rows'] += len(chunk)

        text = chunk['text'].astype('string')
        label = pd.to_numeric(chunk['generated'], errors='coerce')

        audit['missing_text'] += int(text.isna().sum())
        audit['short_text'] += int((text.notna() & text.str.strip().str.len().le(10)).sum())
        audit['invalid_label'] += int((~label.isin([0, 1])).sum())

        clean = valid_rows(chunk)
        counts.update(clean['generated'].astype(int).tolist())

    if not counts:
        raise ValueError('Không có bản ghi hợp lệ với nhãn 0 hoặc 1.')

    total = sum(counts.values())
    audit['valid_rows'] = total

    # Xác định số lượng mẫu mục tiêu cho từng lớp
    if sample_size is None:
        targets = dict(counts)
    else:
        n_target = min(int(sample_size), total)
        targets = {label: int(n_target * count / total) for label, count in counts.items()}
        # Bù phần dư do làm tròn số nguyên
        remainder = n_target - sum(targets.values())
        for label in sorted(counts, key=lambda x: counts[x], reverse=True):
            if remainder <= 0:
                break
            if targets[label] < counts[label]:
                targets[label] += 1
                remainder -= 1

    # Pass 2: Lấy mẫu phân tầng ngẫu nhiên từ từng khối
    parts = []
    for i, chunk in enumerate(pd.read_csv(path, usecols=['text', 'generated'], chunksize=chunksize)):
        clean = valid_rows(chunk)
        for label, target in targets.items():
            if target == 0:
                continue
            group = clean[clean['generated'] == label]
            if group.empty:
                continue
            n = min(len(group), max(1, int(np.ceil(target * len(group) / counts[label]))))
            parts.append(group.sample(n=n, random_state=seed + i * 10 + int(label)))

    sample = pd.concat(parts, ignore_index=True)

    # Khớp chính xác số lượng mẫu mục tiêu
    final_parts = []
    for label, target in targets.items():
        group = sample[sample['generated'] == label]
        if len(group) < target:
            raise ValueError(f'Không đủ mẫu lớp {label}: cần {target}, nhưng chỉ lấy được {len(group)}.')
        if target > 0:
            final_parts.append(group.sample(n=target, random_state=seed + int(label)))

    sample = pd.concat(final_parts, ignore_index=True)
    sample = sample.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Gán nhãn phân loại dạng chuỗi để tiện trực quan
    sample['label'] = sample['generated'].map({0: 'Human', 1: 'AI'})

    return sample, dict(audit), dict(counts)


SAMPLE_SIZE = 60_000
started = time.perf_counter()
df, audit, class_counts = inspect_and_sample(DATA_PATH, SAMPLE_SIZE, seed=RANDOM_STATE)
print(f"⏱️ Đã đọc {audit['raw_rows']:,} dòng và lấy mẫu phân tầng {len(df):,} dòng trong {time.perf_counter() - started:.1f} giây.")


---
## 4. Đánh giá Chất lượng Dữ liệu & Phân bố Nhãn
Kiểm tra các chỉ số kiểm toán trên toàn bộ file CSV và kiểm tra bài viết trùng lặp (`duplicate text`).


In [ ]:
full_counts = pd.Series(class_counts).sort_index()
full_counts.index = full_counts.index.map({0: 'Human', 1: 'AI'})

# Bảng tổng hợp chất lượng dữ liệu
overview = pd.DataFrame({
    'Chỉ số kiểm toán': [
        'Tổng dòng trong CSV', 
        'Dòng hợp lệ', 
        'Văn bản thiếu (NaN/Null)', 
        'Văn bản quá ngắn (<= 10 ký tự)', 
        'Nhãn không hợp lệ', 
        'Bài viết trùng lặp trong mẫu EDA'
    ],
    'Giá trị': [
        audit['raw_rows'], 
        audit['valid_rows'], 
        audit['missing_text'], 
        audit['short_text'], 
        audit['invalid_label'], 
        int(df['text'].duplicated().sum())
    ]
})
display(overview)

# Vẽ biểu đồ phân bố nhãn trên toàn bộ dữ liệu
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(full_counts.index, full_counts.values, color=[PALETTE[x] for x in full_counts.index], width=0.5)
ax.set_title('Phân bố Nhãn trên Toàn bộ Tập Dữ liệu (487k dòng)', fontweight='bold')
ax.set_ylabel('Số lượng bài viết')

for bar, n in zip(bars, full_counts.values):
    pct = n / full_counts.sum()
    ax.text(
        bar.get_x() + bar.get_width() / 2, 
        bar.get_height() / 2, 
        f'{n:,}\n({pct:.1%})', 
        ha='center', va='center', color='white', fontweight='bold', fontsize=12
    )

plt.tight_layout()
plt.show()

print("Phân bố nhãn trong mẫu 60,000 dòng được dùng cho EDA:")
display(df['label'].value_counts().rename_axis('label').reset_index(name='sample_count'))


---
## 5. Trích xuất Đặc trưng Phong cách Ngôn ngữ (Stylometry)
Chuyển đổi văn bản thô thành **11 đặc trưng toán học / ngôn ngữ học định lượng**:
- **Độ phong phú từ vựng (Lexical Diversity):** 
  - `unique_word_count`: Số từ đơn nhất không lặp lại.
  - `type_token_ratio` (TTR): $\frac{\text{unique}}{\text{total}}$ (độ đa dạng từ vựng thô).
  - `root_ttr` (Chỉ số Guiraud): $\frac{\text{unique}}{\sqrt{\text{total}}}$ (ít bị phụ thuộc vào độ dài bài viết hơn TTR thô).
- **Cấu trúc & Ký tự:**
  - `avg_word_len`: Độ dài từ trung bình (số chữ cái / số từ).
  - `avg_sentence_len`: Độ dài câu trung bình (số từ / số câu).
  - `punct_density`: Mật độ dấu câu trên 100 từ.
  - `uppercase_ratio`: Tỷ lệ chữ in hoa trên tổng số chữ cái.
  - `digit_density`: Mật độ chữ số trên 100 ký tự.
  - `newline_density`: Mật độ ký tự xuống dòng trên 100 từ.


In [ ]:
def lexical_stats(text):
    """Tính số từ duy nhất, TTR và Root TTR trên văn bản tiếng Anh."""
    tokens = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text.lower())
    n_tokens = max(len(tokens), 1)
    unique = len(set(tokens))
    return unique, unique / n_tokens, unique / np.sqrt(n_tokens)


started = time.perf_counter()

# 1. Đo lường kích thước & mật độ
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.count(r'\S+')
df['letter_count'] = df['text'].str.count(r'[A-Za-z]')
df['sentence_count'] = df['text'].str.count(r'[.!?]+').clip(lower=1)

# 2. Đặc trưng phong cách trung bình
df['avg_word_len'] = df['letter_count'] / df['word_count'].clip(lower=1)
df['avg_sentence_len'] = df['word_count'] / df['sentence_count']
df['punct_density'] = df['text'].str.count(r'[^\w\s]') / df['word_count'].clip(lower=1) * 100
df['uppercase_ratio'] = df['text'].str.count(r'[A-Z]') / df['letter_count'].clip(lower=1) * 100
df['digit_density'] = df['text'].str.count(r'\d') / df['char_count'].clip(lower=1) * 100
df['newline_density'] = df['text'].str.count(r'\n') / df['word_count'].clip(lower=1) * 100

# 3. Độ đa dạng từ vựng
df[['unique_word_count', 'type_token_ratio', 'root_ttr']] = pd.DataFrame(
    df['text'].map(lexical_stats).tolist(), index=df.index
)

print(f"✅ Đã trích xuất xong 11 đặc trưng cho {len(df):,} mẫu trong {time.perf_counter() - started:.1f} giây.")
display(df[['label', 'word_count', 'avg_word_len', 'avg_sentence_len', 'punct_density', 'uppercase_ratio', 'root_ttr']].head())


---
## 6. Phân tích Độ dài Văn bản: Biến gây nhiễu cần kiểm soát (Confounding Factor)
Nếu hai lớp chênh lệch lớn về độ dài, mô hình học máy rất dễ rơi vào bẫy **"học vẹt"** (Shortcut Learning: cứ thấy bài dài đoán là Human, bài ngắn đoán là AI).
- **Đồ thị 1 (ECDF):** Đường phân phối tích lũy số từ giữa 2 nhóm theo thang log.
- **Đồ thị 2 (Bar plot theo Quantile):** Tỷ lệ phần trăm văn bản AI trong 6 nhóm độ dài.


In [ ]:
# Cắt 0.5% bài viết ngoại lai quá dài để hình vẽ không bị méo
length_df = df[df['word_count'].between(1, df['word_count'].quantile(0.995))].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. ECDF Plot
sns.ecdfplot(data=length_df, x='word_count', hue='label', palette=PALETTE, linewidth=2, ax=axes[0])
axes[0].set_xscale('log')
axes[0].set_title('ECDF Số Từ theo Nhãn (Thang Log)', fontweight='bold')
axes[0].set_xlabel('Số từ trong bài (Word count)')
axes[0].set_ylabel('Tỷ lệ tích lũy')

# 2. Tỷ lệ AI theo từng nhóm Quantile độ dài
length_df['length_group'] = pd.qcut(length_df['word_count'], q=6, duplicates='drop')
ai_rate = length_df.groupby('length_group', observed=True)['generated'].mean().mul(100)
ai_rate.plot(kind='bar', color=COLOR_AI, width=0.6, ax=axes[1])
axes[1].axhline(full_counts['AI'] / full_counts.sum() * 100, color='black', linestyle='--', label='Tỷ lệ AI trung bình toàn bộ')
axes[1].set_title('Tỷ lệ Văn bản AI theo từng Nhóm Độ Dài', fontweight='bold')
axes[1].set_xlabel('Nhóm độ dài (Quantiles)')
axes[1].set_ylabel('Tỷ lệ AI (%)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"📊 Trung vị số từ: Human = {df[df['generated']==0]['word_count'].median():.0f} từ | AI = {df[df['generated']==1]['word_count'].median():.0f} từ")


---
## 7. Xếp hạng Mức độ Khác biệt Chuẩn hóa (Cohen's $d$ Effect Size)
**Cohen's $d$** đo lường khoảng cách giữa hai nhóm theo đơn vị là độ lệch chuẩn gộp:
$$d = \frac{\bar{X}_{\text{AI}} - \bar{X}_{\text{Human}}}{S_{\text{pooled}}}$$
- $|d| \ge 0.8$: **Khác biệt rất lớn (Rất mạnh)**.
- $|d| \approx 0.5$: **Khác biệt trung bình**.
- $|d| \le 0.2$: **Khác biệt nhỏ / không đáng kể**.


In [ ]:
style_features = [
    'word_count', 'avg_word_len', 'avg_sentence_len', 'root_ttr', 
    'punct_density', 'uppercase_ratio', 'digit_density', 'newline_density'
]
human = df[df['generated'] == 0]
ai = df[df['generated'] == 1]

def cohens_d(ai_values, human_values):
    pooled_std = np.sqrt((ai_values.var(ddof=1) + human_values.var(ddof=1)) / 2)
    return 0.0 if pooled_std == 0 else (ai_values.mean() - human_values.mean()) / pooled_std

effect_table = pd.DataFrame({
    'feature': style_features,
    'human_median': [human[x].median() for x in style_features],
    'ai_median': [ai[x].median() for x in style_features],
    'cohens_d_ai_minus_human': [cohens_d(ai[x], human[x]) for x in style_features],
}).sort_values('cohens_d_ai_minus_human')

print("Bảng thống kê Cohen's d và Trung vị của từng đặc trưng:")
display(effect_table.round(3))

# Vẽ biểu đồ thanh ngang Cohen's d
fig, ax = plt.subplots(figsize=(8, 5))
colors = np.where(effect_table['cohens_d_ai_minus_human'] >= 0, COLOR_AI, COLOR_HUMAN)
bars = ax.barh(effect_table['feature'], effect_table['cohens_d_ai_minus_human'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title("Khác biệt chuẩn hóa giữa AI và Human (Cohen's d)", fontweight='bold')
ax.set_xlabel("Cohen's d (Đỏ: AI lớn hơn | Xanh: Human lớn hơn)")

# Thêm nhãn số liệu
for bar, val in zip(bars, effect_table['cohens_d_ai_minus_human']):
    offset = 0.03 if val >= 0 else -0.03
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2, f'{val:.2f}', va='center', ha=ha, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


---
## 8. Thí nghiệm Kiểm chứng: Khống chế Biến Độ dài Bài viết
Để chứng minh sự khác biệt là do **bản chất phong cách viết** chứ không phải do bài viết dài hay ngắn:
- Lọc các bài viết có số từ nằm trong khoảng phân vị $Q_1$ đến $Q_3$ (279 đến 470 từ).
- Lấy mẫu cân bằng tuyệt đối: **10,000 bài Human** và **10,000 bài AI**.
- Vẽ biểu đồ mật độ phân phối xác suất (KDE Plot) cho 4 đặc trưng cốt lõi.


In [ ]:
low_words, high_words = df['word_count'].quantile([0.25, 0.75])
matched = df[df['word_count'].between(low_words, high_words)].copy()
matched_n = min(matched['label'].value_counts().min(), 10_000)
matched = matched.groupby('label', group_keys=False).sample(n=matched_n, random_state=RANDOM_STATE)

features_to_plot = ['avg_word_len', 'root_ttr', 'punct_density', 'uppercase_ratio']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for feature, ax in zip(features_to_plot, axes.ravel()):
    upper = matched[feature].quantile(0.99)
    sns.kdeplot(
        data=matched[matched[feature] <= upper], 
        x=feature, hue='label', 
        common_norm=False, fill=True, alpha=0.25, 
        palette=PALETTE, ax=ax
    )
    ax.set_title(f'{feature} khi độ dài được kiểm soát', fontweight='bold')
    ax.set_ylabel('Mật độ xác suất (Density)')

plt.suptitle(
    f'Thí nghiệm Mẫu Cân bằng: {matched_n:,} bài mỗi lớp, nằm trong khoảng {low_words:.0f}–{high_words:.0f} từ', 
    y=1.02, fontweight='bold', fontsize=14
)
plt.tight_layout()
plt.show()


---
## 9. Khám phá "Dấu vân tay Từ vựng" (TF-IDF Discriminative N-grams)
- Lấy mẫu cân bằng 12,000 bài mỗi lớp.
- Trích xuất 60,000 từ đơn và cụm 2 từ (n-grams: 1–2).
- So sánh hiệu số $\text{TF-IDF}_{\text{AI}} - \text{TF-IDF}_{\text{Human}}$ để tìm:
  - Top 12 cụm từ thiên hướng **Human** dùng nhiều nhất (điểm âm lớn nhất).
  - Top 12 cụm từ thiên hướng **AI** dùng nhiều nhất (điểm dương lớn nhất).


In [ ]:
ngram_n = min(df['label'].value_counts().min(), 12_000)
ngram_df = df.groupby('label', group_keys=False).sample(n=ngram_n, random_state=RANDOM_STATE).reset_index(drop=True)

vectorizer = TfidfVectorizer(
    lowercase=True, stop_words='english', ngram_range=(1, 2),
    min_df=5, max_df=0.90, sublinear_tf=True, max_features=60_000
)
X = vectorizer.fit_transform(ngram_df['text'])
terms = vectorizer.get_feature_names_out()

# Dùng .to_numpy() để tránh lỗi không tương thích phiên bản pandas/scipy
mean_ai = np.asarray(X[ngram_df['generated'].eq(1).to_numpy()].mean(axis=0)).ravel()
mean_human = np.asarray(X[ngram_df['generated'].eq(0).to_numpy()].mean(axis=0)).ravel()

term_scores = pd.DataFrame({'term': terms, 'ai_minus_human_tfidf': mean_ai - mean_human})
top_human = term_scores.nsmallest(12, 'ai_minus_human_tfidf').sort_values('ai_minus_human_tfidf')
top_ai = term_scores.nlargest(12, 'ai_minus_human_tfidf').sort_values('ai_minus_human_tfidf')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(top_human['term'], -top_human['ai_minus_human_tfidf'], color=COLOR_HUMAN)
axes[0].set_title('N-grams đặc trưng hơn cho Human', fontweight='bold')
axes[0].set_xlabel('Độ lệch TF-IDF (nghiêng về Human)')

axes[1].barh(top_ai['term'], top_ai['ai_minus_human_tfidf'], color=COLOR_AI)
axes[1].set_title('N-grams đặc trưng hơn cho AI', fontweight='bold')
axes[1].set_xlabel('Độ lệch TF-IDF (nghiêng về AI)')

plt.tight_layout()
plt.show()

print("Bảng tổng hợp Top N-grams đặc trưng nhất:")
display(pd.concat([top_human.assign(class_hint='Human'), top_ai.assign(class_hint='AI')], ignore_index=True))


---
## 10. Kiểm tra Định tính & Đúc kết Chiến lược Triển khai
Đọc ngẫu nhiên 1 đoạn văn bản mẫu của mỗi bên để đối chiếu cảm quan ngôn ngữ học, sau đó tổng kết toàn bộ kiến trúc cho các notebook tiếp theo.


In [ ]:
# Trích xuất 1 bài mẫu Human và 1 bài mẫu AI
examples = (
    df.groupby('label', group_keys=False).sample(n=1, random_state=123)
    [['label', 'word_count', 'avg_word_len', 'punct_density', 'text']]
    .assign(preview=lambda x: x['text'].str.slice(0, 500) + '...')
    [['label', 'word_count', 'avg_word_len', 'punct_density', 'preview']]
    .reset_index(drop=True)
)
display(examples)

strongest = effect_table.loc[effect_table['cohens_d_ai_minus_human'].abs().idxmax()]

print("""
================================================================================
📌 TỔNG KẾT EDA & QUY TẮC THIẾT KẾ MÔ HÌNH:
================================================================================
1. ĐẶC TRƯNG THEN CHỐT:
   - avg_word_len là vũ khí mạnh nhất (Cohen's d = +1.07). AI dùng từ dài và học thuật hơn rõ rệt.
   - root_ttr (độ đa dạng từ vựng) gần như trùng khít (d = -0.03) -> LOẠI BỎ vì là nhiễu.
   - word_count (số từ) là biến gây nhiễu -> TUYỆT ĐỐI KHÔNG dùng làm feature độc lập.

2. ĐỊNH HƯỚNG 4 PIPELINE HUẤN LUYỆN:
   - Pipeline 1: Word TF-IDF + Logistic Regression / Linear SVM làm baseline nhẹ.
   - Pipeline 2: Word + Character TF-IDF (3-5 n-grams) -> ỨNG VIÊN ƯU TIÊN DEPLOY CHROME EXTENSION.
                 Bắt trọn hậu tố dài của AI và nhịp điệu dấu câu mà không phụ thuộc chủ đề.
   - Pipeline 3: Stylometry (chỉ giữ avg_word_len, newline, punct, digit) + LightGBM / XGBoost.
   - Pipeline 4: Fine-tuned Transformer (DeBERTa-v3 / RoBERTa trên GPU T4) để benchmark ngữ cảnh sâu.

3. QUY TẮC ĐÁNH GIÁ (NO DATA LEAKAGE):
   - Giữ riêng 15% Test Set. Dùng Stratified 5-Fold Cross Validation trên 85% Train.
   - Chọn threshold tối ưu trên Validation/OOF, không tối ưu trực tiếp trên Test.
================================================================================
""")
